📝
# FieldCare — Sprint 3 Live notebook

Use this **one Live notebook for all four live sessions in this sprint**. Your Campus lessons have their own Campus notebook. Save a personal Drive copy of each and return to the same copies throughout the sprint.

HelioDesk’s FieldCare service is our shared case. The workshops let you apply the taught patterns, inspect actual observations and defend your decisions with a peer. Each session below names its starting state, task, evidence and debrief.

## 1. Prepare the runtime and preserve your work

1. Select a Python CPU runtime and run setup and imports once per runtime.
2. Upload your current Campus source/evidence checkpoint in Colab’s Files panel. Set `CAMPUS_CHECKPOINT` to its path. A checkpoint is your saved service files, not the notebook itself.
3. Run the workspace cell. It restores into an isolated Live workspace, preserving the Campus copy. With no checkpoint, `USE_DEMO_RECOVERY=True` explicitly selects a labelled public practice state; it does not establish completion of Campus work.
4. Use the table of contents to open the current session. Run its preparation cells before its task. Later sessions build on earlier live sections unless a supplied checkpoint already provides that starting state.
5. Write your own predictions and explanations. Blank decisions stay pending; **Run all does not complete your work**. Public practice never completes the reserved Campus assessment.

Use a new `RUN_NAME` when importing a newer checkpoint; existing work is deliberately preserved. `RESUME_ZIP` reopens an earlier Live export. Never paste keys into cells or outputs. These workshops use demo/transport fixtures by default, not real model output.


## Instructor guides and live decks in Notion

- [LS 9 — Enabling streaming on the scaffold](https://app.notion.com/p/3ea5a55972ad8118aab4c64059db3adf?pvs=204)
- [LS 10 — Structured logging for AI services](https://app.notion.com/p/3ea5a55972ad81f1bc15c7165ae8ebdb?pvs=204)
- [LS 11 — Logs feeding back into evaluation](https://app.notion.com/p/3ea5a55972ad8184826afc7e99770368?pvs=204)
- [LS 12 — Sprint 3 checkpoint](https://app.notion.com/p/3ea5a55972ad811d89a0e8319caa0f50?pvs=204)


In [ ]:
#@title Load the shared course code from GitHub
from pathlib import Path
import os, re, sys, subprocess, tempfile

# Reviewed shared helpers: this immutable revision keeps every lesson on the same source.
GITHUB_REPO = 'https://github.com/richhiey/ai-app-dev_Mod-B.git'
GITHUB_REF = 'fb9d95b1cadcc29972cb9dd7388201aaa5868e1d'
repo_url = os.environ.get('MODULE_B_REPO_URL', GITHUB_REPO).strip()
repo_ref = os.environ.get('MODULE_B_REPO_REF', GITHUB_REF).strip()
destination = Path(os.environ.get('MODULE_B_REPO', '/content/ai-app-dev_Mod-B')).expanduser()
candidates = [destination, Path.cwd(), *Path.cwd().resolve().parents]
REPO = next((p.resolve() for p in candidates if (p/'src/module_b/notebook.py').is_file()), None)

if REPO is None:
    if not re.fullmatch(r'https://github\.com/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+(?:\.git)?', repo_url):
        raise RuntimeError('The course author must set GITHUB_REPO to the published HTTPS GitHub repository URL.')
    if not re.fullmatch(r'[0-9a-fA-F]{40}|refs/tags/[A-Za-z0-9][A-Za-z0-9._/-]*', repo_ref):
        raise RuntimeError('The course author must set GITHUB_REF to a release tag (refs/tags/...) or full commit SHA.')
    if destination.exists():
        raise FileExistsError('The destination already exists without the expected helpers. Preserve it and use a fresh runtime or a new MODULE_B_REPO path.')
    destination.parent.mkdir(parents=True, exist_ok=True)
    # Prepare separately, so a failed download cannot replace existing student work.
    with tempfile.TemporaryDirectory(prefix='module-b-download-', dir=destination.parent) as temporary:
        checkout = Path(temporary)/'checkout'
        subprocess.run(['git','clone','--no-checkout','--depth','1',repo_url,str(checkout)],check=True)
        subprocess.run(['git','-C',str(checkout),'fetch','--depth','1','origin',repo_ref],check=True)
        subprocess.run(['git','-C',str(checkout),'checkout','--detach','FETCH_HEAD'],check=True)
        if not (checkout/'src/module_b/notebook.py').is_file() or not (checkout/'requirements.lock').is_file():
            raise RuntimeError('The selected repository revision does not contain the expected Module B source layout.')
        checkout.rename(destination)
    REPO = destination.resolve()

# Existing local checkouts are reused; setup never pulls, resets or overwrites them.
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO),'-c',str(REPO/'requirements.lock')],check=True)
if str(REPO/'src') not in sys.path:
    sys.path.insert(0,str(REPO/'src'))
from module_b import __version__
print('Shared package ready:', __version__)
print('Source checkout:', REPO)

In [ ]:
import json, time, tempfile, copy
import httpx
from module_b.evaluation import module_a, load_pipeline, design_revision, evaluate_selected
from module_b.observability import FIELDS, safe_record
from module_b.observability_lab import (prepare_observable_workspace, save_guided_file,
    insert_before_security, service, collect_stream, read_logs, probe_checkpoint, review_checkpoint)
from module_b.notebook import table, register_source, revision, export_progress
from module_b.workspace import restore_workspace

In [ ]:
RUN_NAME = 'sprint-3-live'
CAMPUS_CHECKPOINT = ''  # Current Campus Sprint 3 checkpoint, or your completed secured Sprint 2 checkpoint
USE_DEMO_RECOVERY = True
RESUME_ZIP = ''
work = REPO/'work'/RUN_NAME
if RESUME_ZIP and not (work/'service').exists():
    restore_workspace(RESUME_ZIP, work/'service')
if not CAMPUS_CHECKPOINT and not USE_DEMO_RECOVERY and not RESUME_ZIP:
    raise ValueError('Select your checkpoint, or explicitly choose demo recovery.')
project, inherited = prepare_observable_workspace(work/'service',
    checkpoint=CAMPUS_CHECKPOINT or None, repo_root=REPO)
patterns = REPO/'examples/patterns/sprint_3'
evidence = {}
print(inherited['provenance'])
print('Your cumulative source:', project)

## Prepare the visible streaming pattern

The supplied bridge reuses the broader Module A FieldCare application using approved synthetic case identifiers. It adds replay endpoints; it does not change the narrower existing `/v1/diagnose` contract. `case_id` selects an original evaluation fixture. An optional `question` replaces that fixture's text for a controlled experiment; the tag alone does not prove it is still the approved case.

**Action:** copy the reusable event engine once, then register the routes below. **Reason:** the same event source makes buffered and streamed delivery comparable. `yield` hands one encoded event to the response as it becomes available. New routes register **before** the existing guard inventories POST paths. The observation layer is placed outside the guard so rejections receive a server-generated correlation identifier too.

**Expected output:** paths to your saved files. An existing file is kept on rerun. Inspect it before editing if your implementation differs from this worked example.

The protocol is **NDJSON**: one complete JSON object per line. A `delta` carries text; `complete` marks success; `error` marks failure. HTTP status is committed before the whole stream arrives. Read lines as they arrive instead of calling `.json()` on the full streaming response. See [HTTPX streaming responses](https://www.python-httpx.org/quickstart/#streaming-responses).

In [ ]:
# Engine mechanics live in shared source; route and yield remain visible here.
source = (patterns/'stream_routes.py').read_text()
engine = source.split("@router.post('/v1/fieldcare-stream')")[0]
save_guided_file(project, 'app/stream_engine.py', engine)
routes = """from fastapi import APIRouter, Request
from fastapi.responses import StreamingResponse
from app.stream_engine import ReplayRequest, events, pipeline, DESIGN, PIPELINE_REVISION
from module_b.streaming import encode_event
from module_b.evaluation import run_case, design_revision
router = APIRouter()

@router.post('/v1/fieldcare-stream')
async def stream(body: ReplayRequest, request: Request):
    async def lines():
        async for event in events(body, request):
            yield encode_event(event)
    return StreamingResponse(lines(), media_type='application/x-ndjson',
                             headers={'Cache-Control': 'no-store'})

@router.post('/v1/fieldcare-buffered')
async def buffered(body: ReplayRequest, request: Request):
    return {'events': [event async for event in events(body, request)]}

@router.post('/v1/fieldcare-evidence')
async def evidence(body: ReplayRequest, request: Request):
    context, response = run_case(pipeline, body.case_id, body.question)
    request.state.observation.update(source='module_a_deterministic',
                                    model='module-a-rules', case_id=body.case_id)
    return {'response': response, 'design_revision': PIPELINE_REVISION}
"""
save_guided_file(project, 'app/stream_routes.py', routes)
# This exact file is read by BOTH the service and the original evaluator.
design_path = save_guided_file(project, 'data/pipeline_design.json',
    json.dumps(module_a.default_pipeline_design(), indent=2))
registration = '\nfrom app.stream_routes import router as stream_router\napp.include_router(stream_router)\n'
insert_before_security(project, registration)
# Safe minimal logging is installed now; inspect its fields in the next section.
log_source = "LOG_FIELDS = " + repr(FIELDS) + '\n'
save_guided_file(project, 'app/log_settings.py', log_source)
attachment = (patterns/'attachment.py.txt').read_text()
print(attachment)
register_source(project, attachment)
print('Saved routes:', project/'app/stream_routes.py')

# Live workshops — LS09–12

One Live notebook covers all four workshops. Run `setup`, `imports`, `workspaces` and the visible `stream-build` cell above first. The Campus assessment remains in your separate Campus notebook. Bring your initial Campus checkpoint to the final evidence review, then revise it independently.

Keep your incoming security policy. Delivery comparisons below use separate fresh workers so their calls do not consume the quota experiment's capacity; restarts are **not** window-renewal evidence. The bounded checkpoint probe supports 1–20 admissions in windows of at least 30 seconds. If your policy differs, retain it and design a matching probe with your instructor.

Use only synthetic markers. Default mode makes no provider call. Measurements describe local HTTP delivery of a delayed transport fixture, not production latency or answer quality. Apply/run flags remain false until you make the requested decision. Blank reasoning remains your work.

In [ ]:
from module_b.live_sessions import save_record, read_records
from module_b.notebook import revision
from module_b.observability_lab import registered_routes

# The teaching operations stay visible: buffered POST, streamed lines and header ID.
def live_delivery(case_id, fault='none', marker=None):
    payload = {'case_id': case_id, 'fault': fault}
    if marker is not None:
        payload['question'] = marker
    measurements = {}; logs = []; loaded_designs = []
    # Fresh workers isolate delivery experiments from the inherited caller quota.
    for mode in ['buffered', 'streamed']:
        with tempfile.TemporaryDirectory() as temporary:
            path = Path(temporary)/'safe.jsonl'
            server, keys = service(project, path)
            with server, httpx.Client(base_url=server.base_url, trust_env=False, timeout=20) as client:
                headers = {'X-API-Key': next(iter(keys.values()))}
                started = time.perf_counter()
                if mode == 'buffered':
                    response = client.post('/v1/fieldcare-buffered', json=payload, headers=headers)
                    elapsed = (time.perf_counter()-started)*1000
                    events = response.json().get('events', [])
                    loaded_designs.extend(e['design_revision'] for e in events if e.get('type') == 'complete' and 'design_revision' in e)
                    measurements[mode] = {'status_code': response.status_code,
                        'request_id': response.headers.get('x-request-id'),
                        'first_content_ms': elapsed, 'completion_ms': elapsed,
                        'event_types': [e.get('type') for e in events],
                        'completed': bool(events) and events[-1].get('type') == 'complete'}
                else:
                    with client.stream('POST', '/v1/fieldcare-stream', json=payload, headers=headers) as response:
                        result = collect_stream(response, started)
                    loaded_designs.extend(e['design_revision'] for e in result['events'] if e.get('type') == 'complete' and 'design_revision' in e)
                    measurements[mode] = {k:v for k,v in result.items() if k != 'events'}
                logs.extend(read_logs(path, expected=1))
    return {'evidence_kind': 'actual_local_http_transport_fixture', 'case_id': case_id,
        'source_revision': revision(project),
        'design_revision': loaded_designs[0] if loaded_designs and len(set(loaded_designs)) == 1 else None,
        'loaded_design_revisions': loaded_designs,
        'experiments': 'two separate fresh delivery workers; no renewal claim',
        'measurements': measurements, 'logs': logs}

print('Source:', project)
print('Inherited security configuration:')
print((project/'app/security_settings.py').read_text())


## LS09 — Client progress and honest completion

Inspect `stream-build`: locate `yield encode_event(event)`, the buffered collection, and the registration before the security inventory. Predict which client can display partial progress. Run the guided comparison, then inject an interruption. At minutes 12 and 30, pause and explain one observation before continuing.

**Independent task · 12 minutes:** Choose the other public case (`EVAL-FC-003`), predict normal versus interrupted terminal evidence, then run both. Keep one buffered consumer and name where partial display helps a human. Use timings and terminal events to defend your choice; do not claim improved quality or faster total completion.

In [ ]:
ls09_normal = live_delivery('EVAL-FC-005')
ls09_interrupted = live_delivery('EVAL-FC-005', fault='interrupt')
for label, observation in [('normal', ls09_normal), ('interrupted', ls09_interrupted)]:
    print(label)
    table([{'delivery': mode, **row} for mode, row in observation['measurements'].items()])
# HTTP 200 is the header status. The final event can still be error.


In [ ]:
RUN_LS09_TASK = False
LS09_CASE = ''  # Choose EVAL-FC-003 for the changed-case task.
ls09_prediction = ''
ls09_task = {}
if RUN_LS09_TASK:
    assert LS09_CASE == 'EVAL-FC-003'
    ls09_task = {'normal': live_delivery(LS09_CASE),
                 'interrupted': live_delivery(LS09_CASE, fault='interrupt')}
    for name, observation in ls09_task.items():
        print(name); table([{'delivery': k, **v} for k,v in observation['measurements'].items()])
ls09_reasoning = ''  # Client, timing, terminal state, retained buffered use and limit.
save_record(project, 9, observations={'guided': ls09_normal, 'interrupted': ls09_interrupted,
    'independent': ls09_task}, reasoning={'interpretation': ls09_reasoning}, origin='learner_service')


## LS10 — A useful record without private content

Minimal safe logging was attached in section 3 so earlier observations were safe. Now inspect its actual boundary, choose the fields deliberately, and run the normal/error paths again. Required teaching fields include request ID, model, tokens, latency and errors; add outcome, source and case ID to interpret them correctly. `tokens=None` means unknown, not zero.

**Independent task · 12 minutes:** Configure the required field set, choose a fresh nonfunctional `SYNTHETIC-...` marker, and prove both its absence from saved output and the usefulness of what remains. Follow a request ID from the client header into the terminal record. Inspect invalid, unauthorized and interrupted paths as well as normal completion. The supplied allowlist is schema minimization, not a universal personal-data detector.

In [ ]:
print((patterns/'attachment.py.txt').read_text())
print((project/'app/log_settings.py').read_text())
print('Required fields:', FIELDS)
APPLY_LS10_FIELDS = False
LS10_FIELDS = None  # Write your explicit tuple after inspecting the required fields.
if APPLY_LS10_FIELDS:
    assert LS10_FIELDS is not None and set(FIELDS) <= set(LS10_FIELDS)
    assert not set(LS10_FIELDS) - set(FIELDS), 'Do not add raw prompt, headers or exceptions.'
    from module_b.edits import write_source
    target = project/'app/log_settings.py'
    write_source(project, 'app/log_settings.py', 'LOG_FIELDS = '+repr(tuple(LS10_FIELDS))+'\n',
                 expected=target.read_text())
# Inspect actual saved records, not just what the screen chose to display.
table(ls09_normal['logs'] + ls09_interrupted['logs'])


In [ ]:
RUN_LS10_TASK = False
LS10_MARKER = ''  # A fresh synthetic marker; never a real person's data or key.
ls10_task = {}
if RUN_LS10_TASK:
    assert LS10_MARKER.startswith('SYNTHETIC-') and len(LS10_MARKER) >= 16
    ls10_task = probe_checkpoint(project, case_id='EVAL-FC-005',
        companion='EVAL-FC-003', marker=LS10_MARKER)
    table(ls10_task['logs'], ['request_id','case_id','status_code','outcome','model','tokens','latency_ms','error_category'])
    print('Marker absent:', ls10_task['marker_absent'], 'Useful records:', ls10_task['safe_metadata_present'])
    print('Experimental boundaries:', ls10_task['experiments'])
ls10_reasoning = ''  # Correlation example, exclusion + retained fields, unknown usage, limits.
save_record(project, 10, observations={'independent': ls10_task,
    'field_configuration': (project/'app/log_settings.py').read_text()},
    reasoning={'interpretation': ls10_reasoning}, origin='learner_service')


## LS11 — From an operational signal to an original evaluation

A failed stream can suggest a useful investigation. It cannot establish that the answer violated a quality criterion. Keep these links explicit: **request ID → case ID → hypothesis → original criterion → design revision → evaluator run**. Reuse the frozen Module A evaluator and the design loaded by the service. A question override is a different input; its response is not automatically the original evaluation case.

The guided pair is `EVAL-FC-005` with companion `EVAL-FC-003`. Their default design passes. A separate temporary regression demonstrates a real failure and restoration without altering your service.

**Independent task · 12 minutes:** Select `EVAL-FC-001` as a different approved public case and `EVAL-FC-003` as companion. Generate a fresh service observation without a question override. State a hypothesis, inspect the original criteria, run the original evaluator and interpret an honest pass or failure. Keep the shared design unless the evidence justifies a change. A pass is a valid result; it does not certify live-provider wording.

In [ ]:
LS11_GUIDED = ['EVAL-FC-005', 'EVAL-FC-003']
pipeline = load_pipeline(design_path)
criteria = [module_a.eval_case_by_id(pipeline.env, c) for c in LS11_GUIDED]
table(criteria, ['eval_id','required_doc_ids','required_tool_calls','expected_response_flags'])
# Authentic Module A criteria and evaluator: no new scorer or guessed grade.
ls11_before = evaluate_selected(design_path, LS11_GUIDED)
table(ls11_before['rows'], ['eval_id','pipeline_pass','missing_required_docs','missing_expected_flags'])
with tempfile.TemporaryDirectory() as temporary:
    experiment = Path(temporary)/'public-regression.json'
    original = design_path.read_text(); changed = json.loads(original)
    changed['response_policy']['ask_before_model_specific_guidance_when_ids_missing'] = False
    experiment.write_text(json.dumps(changed))
    ls11_regression = evaluate_selected(experiment, LS11_GUIDED)
    experiment.write_text(original)
    ls11_restored = evaluate_selected(experiment, LS11_GUIDED)
table(ls11_regression['rows'], ['eval_id','pipeline_pass','missing_expected_flags'])
table(ls11_restored['rows'], ['eval_id','pipeline_pass','missing_expected_flags'])


In [ ]:
RUN_LS11_TASK = False
LS11_CASE = ''; LS11_COMPANION = ''
ls11_hypothesis = ''
ls11_task = {}
if RUN_LS11_TASK:
    assert [LS11_CASE, LS11_COMPANION] == ['EVAL-FC-001', 'EVAL-FC-003']
    signal = live_delivery(LS11_CASE)  # No question override: original case input.
    original_cases = [module_a.eval_case_by_id(load_pipeline(design_path).env, c)
                      for c in [LS11_CASE, LS11_COMPANION]]
    table(original_cases, ['eval_id','required_doc_ids','required_tool_calls','expected_response_flags'])
    rerun = evaluate_selected(design_path, [LS11_CASE, LS11_COMPANION])
    assert signal['design_revision'] == rerun['design_revision']
    ls11_task = {'signal': signal, 'hypothesis': ls11_hypothesis, 'criteria': original_cases,
                 'evaluation': rerun, 'question_override': False}
    table(rerun['rows'], ['eval_id','pipeline_pass','missing_required_docs','missing_expected_flags'])
ls11_reasoning = ''  # Actual criterion/result, config identity and what this cannot establish.
save_record(project, 11, observations={'baseline': ls11_before,
    'temporary_regression': ls11_regression, 'restored': ls11_restored, 'independent': ls11_task},
    reasoning={'interpretation': ls11_reasoning}, origin='learner_service')


## LS12 — Demonstrate and defend the evidence chain

Triads rotate owner, boundary reviewer and traceability reviewer. Each 8-minute turn: 4-minute demo, 2-minute challenge, 1-minute feedback, 1-minute record. Show one short actual stream; use dated saved records for broader checks. Demonstrate normal and partial states, useful minimized logs, a real original-evaluator run, retained security and buffered behavior.

**Independent repair · 12 minutes:** Repair one evidence gap or narrow an unsupported claim, rerun the relevant observation and record the residual limitation. The optional public checkpoint below collects a broad local fixture check; it is not the reserved Campus assessment and never awards a grade. Use your LS11 public pair for traceability. No provider or hosted-runtime claim follows from this demo.

In [ ]:
RUN_LS12_CHECK = False
ls12_observations = {}
if RUN_LS12_CHECK:
    assert ls11_task, 'Complete the public LS11 case selection first.'
    ls12_observations = probe_checkpoint(project, case_id=LS11_CASE,
        companion=LS11_COMPANION, marker='SYNTHETIC-LS12-LOCAL-PRIVATE')
    ls12_evaluation = evaluate_selected(design_path, [LS11_CASE, LS11_COMPANION])
    assert ls12_evaluation['design_revision'] == ls12_observations['design_revision']
    ls12_observations['original_evaluation'] = ls12_evaluation
    print({k:v for k,v in ls12_observations.items() if k not in ('logs','original_evaluation')})
ls12_reasoning = ''  # Peer challenge, corrected claim, checked repair and residual limitation.
save_record(project, 12, observations=ls12_observations,
    reasoning={'interpretation': ls12_reasoning}, origin='learner_service')
evidence['live_workshops'] = read_records(project)
print('Live records saved. Use the single final export below; Campus assessment remains separate.')


## Save your Live evidence and carry improvements back

Run this export after your session and again after later changes. It saves the Live workspace and the four public workshop records; it does not mark your Campus assessment complete. Download the printed ZIP from Colab’s Files panel and save this Live notebook separately.

**Apply a chosen improvement to Campus:** open your Campus copy, compare the changed source file with your own file, and apply only the change you can explain using the Campus editing cell or workspace editor. Rerun the affected route/control checks and the preservation checks. Then export a new Campus checkpoint. Do not replace the entire Campus project with a worked Live example. Your next sprint starts from the latest verified **Campus** checkpoint.

**Expected result:** the ZIP contains your Live source and safe observations, all session records remain available on resume, and your Campus workspace has not been changed by this notebook.


In [ ]:
from module_b.live_sessions import read_records
final_project = project
final_stage = 'live-limit' if final_project.name == 'live-limit' else 'auth' if final_project.name == 'auth' else 'live-project'
live_records = {**read_records(project), **read_records(final_project)}
(final_project/'fixtures/live-session-records.json').write_text(json.dumps(live_records, indent=2)+'\n')
readiness = {'structural_ready': False, 'written_evidence_present': False,
    'scope': 'live_formative_only', 'campus_assessment': 'not_evaluated',
    'session_review_status': {k:v['review_status'] for k,v in live_records.items()}}
SPRINT3_EXPORT = export_progress(final_project, REPO/'artifacts'/f'sprint-3-live', sprint=3,
    evidence={'live_sessions': live_records}, notes={'purpose':'Live workshop evidence; apply chosen changes back to Campus explicitly.'}, readiness=readiness)
print('Resume stage:', final_stage)
print('Live review status:', readiness['session_review_status'])


## Watch out for

| Observation | Check and recovery |
|---|---|
| Your new checkpoint was not imported | Existing Live work is preserved. Choose a new `RUN_NAME`, then rerun the workspace cell. |
| A later session has no prepared service | Run the named earlier guided cells or import the correct checkpoint; do not infer completion from a blank task. |
| A task still says pending | Fill your decisions and reasoning, execute its checks, then export again. |
| Runtime ended before saving | Restore your downloaded Live ZIP in a fresh runtime; saving only the notebook does not preserve workspace files. |

## Quick reference

| Action | Location |
|---|---|
| Shared code and dependency installation | `setup` |
| Import Campus or resume Live work | `workspaces` |
| Session activity and evidence | Named LS section |
| Save source and public observations | `save-checkpoint` |

## Summary

You have a separate place to rehearse, investigate and discuss FieldCare service behavior. Keep the observations tied to the source you ran, carry justified changes back into your Campus project, and preserve both your notebook and source/evidence checkpoint.
